# Ingestion and candidate generation
Run from the repository root. This notebook makes the synthetic fixture, ingests it, resolves its isolated oracle, and builds Label Studio tasks.

In [ ]:
from pathlib import Path

from autolabeler.fixture import create_fixture
from autolabeler.ingest import SplitConfig, ingest_pdf
from autolabeler.tasks import generate_tasks, resolve_oracle

create_fixture("data/demo", pages=20)
manifest = ingest_pdf("data/demo/demo.pdf", "workspace", "v1", SplitConfig(max_chars=1800, overlap_chars=200))
manifest

In [ ]:
run = Path("workspace") / manifest["document_id"] / manifest["pipeline_version"]
tasks = generate_tasks(
    str(run / "chunks.jsonl"),
    "data/demo/queries.jsonl",
    "workspace/tasks.json",
    k=5,
    ranking_output="workspace/ranking.jsonl",
)
oracle = resolve_oracle(str(run / "chunks.jsonl"), "data/demo/oracle_seed.jsonl", "data/demo/oracle.jsonl")
len(tasks), tasks[:1], oracle

Import `workspace/tasks.json` into a Label Studio project configured with `label_studio/retrieval.xml`. Keep `oracle.jsonl` away from annotators and retrieval code.